# McGill COMP551, MINI-PROJECT 3: Odd-One-Out Image Groups

**Kaggle team name**

## Overview

5 grayscale images per group; 4 share a property, 1 is the outlier. Task: predict outlier index (0–4).

In [ ]:
import os
from pathlib import Path

for _root in [Path.cwd(), *Path.cwd().resolve().parents]:
    if (_root / "datasets" / "x_train.npy").is_file():
        os.chdir(_root)
        break
else:
    raise FileNotFoundError("datasets/x_train.npy not found — open the 551-A3 folder and restart the kernel.")

import numpy as np
import matplotlib.pyplot as plt

x = np.load("datasets/x_train.npy")
y = np.load("datasets/y_train.npy")
print(f"Train: {x.shape[0]} groups, 5 images each, {x.shape[2]}x{x.shape[3]}")


## Step 1 — Data & Preprocessing

Preprocessing: crop to object, center in 32×32, extract shape metadata (Hu moments, contours, PCA).

In [2]:
!python -m src.preprocess --data-dir datasets

preprocessing train...
processing group 1/3000
processing group 200/3000
processing group 400/3000
processing group 600/3000
processing group 800/3000
processing group 1000/3000
processing group 1200/3000
processing group 1400/3000
processing group 1600/3000
processing group 1800/3000
processing group 2000/3000
processing group 2200/3000
processing group 2400/3000
processing group 2600/3000
processing group 2800/3000
processing group 3000/3000
preprocessing test...
processing group 1/2000
processing group 200/2000
processing group 400/2000
processing group 600/2000
processing group 800/2000
processing group 1000/2000
processing group 1200/2000
processing group 1400/2000
processing group 1600/2000
processing group 1800/2000
processing group 2000/2000
computing train-only normalization stats...
applying normalization...
saving outputs...

Done.
Saved:
  datasets/processed_train_area_only.npz
  datasets/processed_test_area_only.npz
  datasets/preprocess_stats_area_only.npz
  datasets/prep

## Step 2 — Model

**OddOneOutNet:** TinyCNN + meta MLP → set attention + archetype clustering + pairwise relations → 5-way classifier.

**Params:** < 25,000.

In [ ]:
import os
import sys
from pathlib import Path

for _p in [Path.cwd(), *Path.cwd().resolve().parents]:
    if (_p / "src" / "nets.py").is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        REPO = _p
        break
else:
    raise RuntimeError("File → Open Folder → select the 551-A3 folder, then restart the kernel.")

os.chdir(REPO)

import numpy as np
import torch
from src.nets import OddOneOutNet, load_checkpoint_or_raise

z = np.load(REPO / "datasets/processed_train_area_only.npz", allow_pickle=True)
meta_dim = z["meta"].shape[-1]
model = OddOneOutNet(meta_dim=meta_dim)
ckpt = load_checkpoint_or_raise(model)
print(f"Loaded weights from {ckpt}")

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")
assert total_params <= 25_000, f"Model too large: {total_params:,} > 25,000"


## Step 3 — Training (optional)

Run to retrain. Otherwise we use the saved `best_model.pt`.

In [ ]:
# Uncomment to retrain:
# !python -m src.nets --data-dir datasets

## Step 4 — Public Test Accuracy

In [ ]:
try:
    REPO
except NameError:
    import os
    import sys
    from pathlib import Path
    for _p in [Path.cwd(), *Path.cwd().resolve().parents]:
        if (_p / "src" / "nets.py").is_file():
            REPO = _p
            if str(_p) not in sys.path:
                sys.path.insert(0, str(_p))
            break
    else:
        raise RuntimeError("Run Step 2 first.")
    os.chdir(REPO)

import numpy as np
import torch
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader
from src.nets import DictDataset, load_npz_arrays, predict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Default test-time augmentation is 8×; on CPU that is very slow — use 1× on CPU.
_tta = 8 if torch.cuda.is_available() else 1

test_arrays = load_npz_arrays(str(REPO / "datasets/processed_test_area_only.npz"))
y_test = np.load(REPO / "datasets/y_test.npy")
n = len(y_test)
pub_arrays = {k: v[:n] for k, v in test_arrays.items() if isinstance(v, np.ndarray)}
pub_ds = DictDataset(pub_arrays, labels=y_test)
pub_loader = DataLoader(pub_ds, batch_size=64, shuffle=False)

pub_preds = predict(model, pub_loader, device, tta_perms=_tta)
acc = accuracy_score(y_test, pub_preds)
print(f"Public test accuracy (must match Kaggle public leaderboard): {acc * 100:.2f}%")


## Step 5 — Kaggle Submission CSV

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from src.nets import DictDataset, generate_csv_kaggle, predict

_tta = 8 if torch.cuda.is_available() else 1

test_ds = DictDataset(test_arrays, labels=None)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)
yh_test = predict(model, test_loader, device, tta_perms=_tta)

generate_csv_kaggle(yh_test)
print(f"Saved {len(yh_test)} predictions to predicted_labels.csv")
